# 第 10 讲：Inference

这是 CS336 Lecture 10 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_10.py`
- 官方形式：executable lecture
- 英文标题：Inference

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲讲模型真正被使用时的系统问题：prefill/decode、TTFT、latency、throughput、KV cache、GQA/MLA、quantization、speculative sampling、paged attention。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

先理解 inference workload，再看 lossy shortcut、lossless shortcut 和动态请求调度。


In [1]:
lecture = 10
title = 'Inference'
source = 'external/cs336-lectures/lecture_10.py'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 10: Inference
source: external/cs336-lectures/lecture_10.py


## 2. Prefill 与 decode

prefill 并行处理 prompt，decode 逐 token 生成，decode 更容易 memory-bound。


## 3. 指标

TTFT 看首 token 等待，latency 看单请求速度，throughput 看批量处理能力。


## 4. KV cache

缓存历史 K/V 避免重复计算，但长上下文和大 batch 会让 KV cache 成为显存瓶颈。


## 5. 压缩和捷径

GQA/MLA、quantization、pruning/distillation 都在减少 inference 成本。


## 6. 动态 workload

连续 batching、paged attention 解决真实服务里的变长请求和碎片化。


## 动手实验 1：KV cache 大小估算

先直接运行，再改一个数字或字符串。


In [2]:
def kv_cache_gb(layers, seq, batch, kv_heads, head_dim, bytes_per_value=2):
    # K and V
    return layers * seq * batch * kv_heads * head_dim * 2 * bytes_per_value / 1e9

for seq in [4096, 32768, 131072]:
    print(seq, "tokens:", kv_cache_gb(32, seq, 8, 8, 128), "GB")


4096 tokens: 4.294967296 GB
32768 tokens: 34.359738368 GB
131072 tokens: 137.438953472 GB


## 动手实验 2：TTFT、latency、throughput 的区别

先直接运行，再改一个数字或字符串。


In [3]:
requests = [
    {"prefill_s": 0.8, "decode_tokens": 100, "decode_s": 5.0},
    {"prefill_s": 0.2, "decode_tokens": 20, "decode_s": 1.0},
]
for r in requests:
    print("TTFT", r["prefill_s"], "s")
    print("latency/token", r["decode_s"] / r["decode_tokens"], "s/token")


TTFT 0.8 s
latency/token 0.05 s/token
TTFT 0.2 s
latency/token 0.05 s/token


## 动手实验 3：Speculative sampling 接受率玩具模型

先直接运行，再改一个数字或字符串。


In [4]:
draft_tokens = 1000
accept_rate = 0.7
target_calls_without = draft_tokens
target_calls_with = draft_tokens * (1 - accept_rate)
print("target calls without speculation:", target_calls_without)
print("extra target work after accepted drafts:", target_calls_with)


target calls without speculation: 1000
extra target work after accepted drafts: 300.00000000000006


## 今日检查点

1. 能区分 prefill 和 decode。
2. 能解释 KV cache 为什么会占很多显存。
3. 能说出 TTFT、latency、throughput 分别适合衡量什么。
4. 能理解 speculative sampling 为什么需要校验。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
